In [20]:
import gensim.downloader as api

wv = api.load("glove-wiki-gigaword-100")  # small + fast

[==================================================] 100.0% 128.1/128.1MB downloaded


In [21]:
wv.similarity(w1="great", w2="good")

np.float32(0.7592797)

In [22]:
wv_great = wv["great"]
wv_good = wv["good"]

In [23]:
wv_great.shape, wv_good.shape

((100,), (100,))

In [24]:
import pandas as pd

#read the dataset with name "Fake_Real_Data.csv" and store it in a variable df
df_fake = pd.read_csv("fake-and-real-news-dataset/versions/1/Fake.csv")

#print the shape of dataframe
print(df_fake.shape)

#print top 5 rows
df_fake.head(5)

(23481, 4)


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [25]:
#read the dataset with name "Fake_Real_Data.csv" and store it in a variable df
df_true = pd.read_csv("fake-and-real-news-dataset/versions/1/True.csv")

#print the shape of dataframe
print(df_true.shape)

#print top 5 rows
df_true.head(5)

(21417, 4)


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [26]:
df_true['label'] = 0
df_true.sample(5)

,title,text,subject,date,label
1440,"The press, branded the 'enemy' by Trump, incre...",NEW YORK (Reuters) - Americans are increasing...,politicsNews,"October 3, 2017",0
16574,Indonesia arrests nine with alleged Islamic St...,JAKARTA (Reuters) - Indonesian authorities arr...,worldnews,"October 25, 2017",0
17094,"Venezuela opposition refuses swearing in, smal...","CARACAS/PUERTO ORDAZ, Venezuela (Reuters) - Ve...",worldnews,"October 18, 2017",0
20244,"Brazil police suspect Temer, aides involved in...",BRASILIA/SAO PAULO (Reuters) - Brazil s federa...,worldnews,"September 12, 2017",0
18784,Merkel's conservatives warned not to close off...,"MUNICH, Germany (Reuters) - German Chancellor ...",worldnews,"September 28, 2017",0


In [27]:
df_fake['label'] = 1
df_fake.sample(5)

,title,text,subject,date,label
10605,"DEM PARTY OFFICIAL, Chair Of Black Caucus, Ber...",A Nebraska Democratic Party official has refus...,politics,"Jun 16, 2017",1
9196,SENATOR REVEALS Shocking FBI Corruption in “Wa...,The interview below is pretty much what we all...,politics,"Dec 14, 2017",1
7549,"Rocked By Scandal, Wounded Warrior Project Fi...","On Thursday, two top executives with the Wound...",News,"March 11, 2016",1
6807,"Trends Show Republicans Are SCREWED For 2016,...",Hillary Clinton and Donald Trump emerged victo...,News,"April 20, 2016",1
22852,Boiler Room EP #117 – Straight Outta Tavistock...,Tune in to the Alternate Current Radio Network...,Middle-east,"July 14, 2017",1


In [28]:
df = pd.concat([df_true[:2000], df_fake[:2000]], ignore_index=True)
print(df.shape)
df.sample(5)

(4000, 5)


,title,text,subject,date,label
2747,Trump Has Total UNHINGED Scatter-Brained Ment...,Donald Trump is desperate to distract everyone...,News,"July 25, 2017",1
3334,Confused Old Man Forgets He’s At Arlington Ce...,Donald Trump showed up at Arlington National C...,News,"May 29, 2017",1
971,White House continues to cooperate with specia...,WASHINGTON (Reuters) - Charges brought against...,politicsNews,"October 30, 2017",0
2433,Trump Staffers Admit Their Boss Has No Idea W...,Donald Trump sent a racist elf to make the ann...,News,"September 5, 2017",1
2636,Three Days Before Trump Calls For More Police...,"During his bizarre rally in Youngstown, Ohio...",News,"August 7, 2017",1


In [29]:
df['Text'] = df['title'] + " " + df['text']
df.sample(5)

,title,text,subject,date,label,Text
3222,Trump’s Lawyer’s Response Was Riddled With Er...,Donald Trump hired himself a personal lawyer t...,News,"June 8, 2017",1,Trump’s Lawyer’s Response Was Riddled With Er...
3369,Fox News Staffers Call Sean Hannity MASSIVE E...,"Sean Hannity, in his desperation to show that ...",News,"May 22, 2017",1,Fox News Staffers Call Sean Hannity MASSIVE E...
3559,Sally Yates Just Opened A Can Of Constitution...,During testimony in front of the Senate Intell...,News,"May 8, 2017",1,Sally Yates Just Opened A Can Of Constitution...
2669,Emboldened NRA Threatens New York Times: ‘We’...,"The NRA has a new favorite toy, but there are ...",News,"August 4, 2017",1,Emboldened NRA Threatens New York Times: ‘We’...
3579,WATCH: Teacher Whitesplains His Use Of ‘N***e...,"Just when you think you ve seen everything, a ...",News,"May 5, 2017",1,WATCH: Teacher Whitesplains His Use Of ‘N***e...


In [30]:
df.label.value_counts()

label
0    2000
1    2000
Name: count, dtype: int64

In [38]:
import spacy
nlp = spacy.load("en_core_web_lg") # if this fails then run "python -m spacy download en_core_web_lg" to download that model

def preprocess_and_vectorize(text):
    # remove stop words and lemmatize the text
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
        
    return wv.get_mean_vector(filtered_tokens)

In [39]:
v = preprocess_and_vectorize("Don't worry if you don't understand")
v

array([ 0.00824455,  0.1148466 ,  0.10014525, -0.01590417, -0.1867531 ,
       -0.01693138, -0.19055161, -0.13156646,  0.11157142, -0.06287318,
       -0.05655883,  0.04814307,  0.02319344, -0.05732825, -0.08515857,
       -0.0515773 , -0.08774573,  0.10966119, -0.08039526,  0.06311665,
       -0.04348028,  0.05490522, -0.04981112, -0.11757043, -0.07608508,
        0.00596857, -0.01438856, -0.10697965,  0.09737557, -0.02107832,
       -0.04912352,  0.11810335,  0.00292108, -0.02939533, -0.0282766 ,
       -0.00319967, -0.00414034,  0.07270555,  0.11295398, -0.06414554,
       -0.13241655, -0.0130035 , -0.02308624, -0.13509399, -0.10252795,
       -0.02777304,  0.11581741,  0.00577323, -0.08457497, -0.22086534,
        0.1032379 ,  0.01338345, -0.04633459,  0.12219629,  0.06156217,
       -0.29172766,  0.08066311, -0.05254054,  0.16190298,  0.03909582,
       -0.01172133,  0.16293988, -0.05789497, -0.10555087,  0.14668298,
        0.11625271,  0.08954   ,  0.04959325, -0.03425451, -0.04

In [36]:
wv.get_mean_vector(["worry","understand"], pre_normalize=False)[:3]

array([0.04421075, 0.570505  , 0.49983   ], dtype=float32)

In [37]:
v1 = wv["worry"]
v2 = wv["understand"]

import numpy as np
np.mean([v1,v2],axis=0)[:3]

array([0.04421075, 0.570505  , 0.49983   ], dtype=float32)

In [40]:
df['vector'] = df['Text'].apply(lambda text: preprocess_and_vectorize(text))
df.sample(5)

,title,text,subject,date,label,Text,vector
1444,Oracle Co-CEO questions policies on student visas,SAN FRANCISCO (Reuters) - Oracle Corp (ORCL.N)...,politicsNews,"October 2, 2017",0,Oracle Co-CEO questions policies on student vi...,"[0.0046253917, 0.020002935, 0.03295987, -0.023..."
310,Republican Party backs Senate candidate Moore:...,(Reuters) - The Republican Party will resume f...,politicsNews,"December 5, 2017",0,Republican Party backs Senate candidate Moore:...,"[-0.011583955, 0.038911786, 0.046663936, -0.02..."
2492,"One More Nazi Resigned From Trump’s White, Ho...",Just days after Donald Trump advisor and avowe...,News,"August 26, 2017",1,"One More Nazi Resigned From Trump’s White, Ho...","[-0.005289325, 0.026712185, 0.038133558, -0.02..."
308,Factbox: Trump on Twitter (December 5) - Utah ...,The following statements were posted to the ve...,politicsNews,"December 5, 2017",0,Factbox: Trump on Twitter (December 5) - Utah ...,"[-0.021706453, -0.0063206255, 0.039723076, 0.0..."
2051,Trump Sends Crazy-Time Tweet To The Wrong Acc...,Donald Trump retweeted fake news videos in the...,News,"November 29, 2017",1,Trump Sends Crazy-Time Tweet To The Wrong Acc...,"[-0.0136301955, 0.023005862, 0.04725453, -0.02..."


In [ ]:
from sklearn.model_selection import train_test_split


#Do the 'train-test' splitting with test size of 20% with random state of 2022 and stratify sampling too
X_train, X_test, y_train, y_test = train_test_split(
    df.vector.values, 
    df.label, 
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022,
    stratify=df.label
)

In [ ]:
print("Shape of X_train before reshaping: ", X_train.shape)
print("Shape of X_test before reshaping: ", X_test.shape)


X_train_2d = np.stack(X_train)
X_test_2d =  np.stack(X_test)

print("Shape of X_train after reshaping: ", X_train_2d.shape)
print("Shape of X_test after reshaping: ", X_test_2d.shape)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

#1. creating a GradientBoosting model object
clf = GradientBoostingClassifier()

#2. fit with all_train_embeddings and y_train
clf.fit(X_train_2d, y_train)


#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = clf.predict(X_test_2d)


#4. print the classfication report
print(classification_report(y_test, y_pred))

In [ ]:
test_news = [
    "Michigan governor denies misleading U.S. House on Flint water (Reuters) - Michigan Governor Rick Snyder denied Thursday that he had misled a U.S. House of Representatives committee last year over testimony on Flintâ€™s water crisis after lawmakers asked if his testimony had been contradicted by a witness in a court hearing. The House Oversight and Government Reform Committee wrote Snyder earlier Thursday asking him about published reports that one of his aides, Harvey Hollins, testified in a court hearing last week in Michigan that he had notified Snyder of an outbreak of Legionnairesâ€™ disease linked to the Flint water crisis in December 2015, rather than 2016 as Snyder had testified. â€œMy testimony was truthful and I stand by it,â€ Snyder told the committee in a letter, adding that his office has provided tens of thousands of pages of records to the committee and would continue to cooperate fully.  Last week, prosecutors in Michigan said Dr. Eden Wells, the stateâ€™s chief medical executive who already faced lesser charges, would become the sixth current or former official to face involuntary manslaughter charges in connection with the crisis. The charges stem from more than 80 cases of Legionnairesâ€™ disease and at least 12 deaths that were believed to be linked to the water in Flint after the city switched its source from Lake Huron to the Flint River in April 2014. Wells was among six current and former Michigan and Flint officials charged in June. The other five, including Michigan Health and Human Services Director Nick Lyon, were charged at the time with involuntary manslaughter",
    " WATCH: Fox News Host Loses Her Sh*t, Says Investigating Russia For Hacking Our Election Is Unpatriotic This woman is insane.In an incredibly disrespectful rant against President Obama and anyone else who supports investigating Russian interference in our election, Fox News host Jeanine Pirro said that anybody who is against Donald Trump is anti-American. Look, it s time to take sides,  she began.",
    " Sarah Palin Celebrates After White Man Who Pulled Gun On Black Protesters Goes Unpunished (VIDEO) Sarah Palin, one of the nigh-innumerable  deplorables  in Donald Trump s  basket,  almost outdid herself in terms of horribleness on Friday."
]

test_news_vectors = [preprocess_and_vectorize(n) for n in test_news]
clf.predict(test_news_vectors)

In [ ]:
#finally print the confusion matrix for the best model (GradientBoostingClassifier)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
cm


from matplotlib import pyplot as plt
import seaborn as sn
plt.figure(figsize = (10,7))
sn.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Prediction')
plt.ylabel('Truth')